# 22 - Vecteurs de Witt : l'arithmétique qui relève Frobenius

Le carnet [01](01-corps-finis-borne-hasse.ipynb) a traité $\mathbb{F}_p$ comme un corps : on y calcule, on y compte des points, et Frobenius $x \mapsto x^p$ y est l'identité. La question de ce carnet est celle que Serre pose au chapitre II de ses *Corps locaux*, et dont l'œuvre d'Illusie -- le complexe de de Rham–Witt, dont il figurait au programme de la conférence Serre 100 -- est le prolongement géométrique : **comment relever $\mathbb{F}_p$ en un anneau de caractéristique $0$ sans perdre l'arithmétique ?**

Le relèvement naïf -- garder les entiers $0,\dots,p-1$ et calculer $\bmod p^n$ -- ne respecte pas la structure : on le mesure au §1, avec deux contre-exemples. Les **vecteurs de Witt** donnent le relèvement *fonctoriel* : $W_n(\mathbb{F}_p)$ est un anneau de caractéristique $0$, isomorphe à $\mathbb{Z}/p^n\mathbb{Z}$.

Le fil, en arithmétique `stdlib` pure -- aucun entier ne sort de $\mathbb{Z}$, et **toutes les divisions du carnet sont exactes et vérifiées au lieu d'être supposées** :

1. **le relèvement naïf échoue** -- deux contre-exemples mesurés ;
2. **les composantes fantômes** -- l'isomorphisme qui linéarise addition et multiplication ;
3. **l'anneau $W_n(\mathbb{F}_p)$** -- l'inversion triangulaire, à division exacte ;
4. **les axiomes, vérifiés exhaustivement** -- pas sur un échantillon : sur tout l'anneau ;
5. **Frobenius, Verschiebung, Teichmüller** -- les trois opérateurs, et l'identité $F V = V F = p$ ;
6. **deux constructions, un seul anneau** -- les polynômes fantômes et l'application de Teichmüller se rencontrent sur $\mathbb{Z}/p^n$.

Ce carnet est un sous-grain de l'EPIC [Serre 100 (#16334)](https://github.com/jsboige/CoursIA/issues/16334).

## 1. De $\mathbb{F}_p$ à $\mathbb{Z}/p^n$ : pourquoi le relèvement naïf échoue

La réduction $\mathbb{Z}/p^n\mathbb{Z} \to \mathbb{F}_p$ est un morphisme d'anneaux : elle ne peut donc pas être fautive. Ce qui est fautif, c'est de vouloir lui donner une **section** qui soit elle aussi un morphisme d'anneaux -- c'est-à-dire de choisir, pour chaque classe $a \bmod p$, un représentant $s(a)$ tel que $s(a \cdot b) = s(a)\,s(b)$ et $s(a + b) = s(a) + s(b)$.

Prenons la section la plus naturelle : $s(a) = a$ (l'entier entre $0$ et $p-1$). Elle échoue sur le produit, et elle échoue sur la somme dès que la caractéristique se manifeste : $1 + 1 = 0$ dans $\mathbb{F}_2$, mais $1 + 1 = 2$ dans $\mathbb{Z}/8$.

In [1]:
def section_naive(a, p):
    """Section naturelle de la reduction Z/p^n -> F_p : l'entier 0 <= a < p."""
    return a % p


def ecart(a, b, op, p, n):
    """Compare s(op(a, b)) dans F_p a op(s(a), s(b)) dans Z/p^n."""
    mod = p ** n
    gauche = section_naive(op(a, b) % p, p)                       # la structure, puis le relevement
    droite = op(section_naive(a, p), section_naive(b, p)) % mod   # le relevement, puis la structure
    return gauche, droite


cas = [(2, 2, 3, 2, "produit", lambda u, v: u * v),
       (1, 1, 2, 3, "somme", lambda u, v: u + v),
       (2, 2, 2, 3, "produit", lambda u, v: u * v)]

print(f"{'a':>2} {'b':>2} {'p':>2} {'n':>2} {'operation':>9} | {'s(op)':>5} {'op(s,s)':>7} | verdict")
for a, b, p, n, nom, op in cas:
    gauche, droite = ecart(a, b, op, p, n)
    verdict = "concorde" if gauche == droite else f"DIVERGE dans Z/{p ** n}"
    print(f"{a:>2} {b:>2} {p:>2} {n:>2} {nom:>9} | {gauche:>5} {droite:>7} | {verdict}")

 a  b  p  n operation | s(op) op(s,s) | verdict
 2  2  3  2   produit |     1       4 | DIVERGE dans Z/9
 1  1  2  3     somme |     0       2 | DIVERGE dans Z/8
 2  2  2  3   produit |     0       0 | concorde


### Lecture du résultat

Les deux premières lignes mesurent l'échec : dans $\mathbb{F}_3$, $2 \cdot 2 = 1$, mais $2 \cdot 2 = 4$ dans $\mathbb{Z}/9$ ; dans $\mathbb{F}_2$, $1 + 1 = 0$, mais $1 + 1 = 2$ dans $\mathbb{Z}/8$. La troisième ligne est le contrôle positif du même instrument : pour $2 \cdot 2$ avec $p = 2$, $n = 3$, la section naïve **concorde** -- un test qui ne saurait rougir que sur les cas favorables ne prouverait rien.

Ce que cela dit : il n'existe pas de section qui soit un morphisme d'anneaux *et* qui préserve les coordonnées. La sortie de Witt est de **changer les coordonnées** : les coordonnées de Witt ne sont pas les chiffres, et c'est leur addition qui absorbe les retenues que la section naïve laisse passer.

## 2. Les composantes fantômes : linéariser l'anneau

Les **polynômes de Witt** sont

$$w_i(X_0, \dots, X_i) = \sum_{j=0}^{i} p^j\, X_j^{\,p^{\,i-j}}$$

et leur vertu est de transformer les opérations de l'anneau en opérations *composante par composante* : pour tous vecteurs $a, b$,

$$w_i(a +_W b) = w_i(a) + w_i(b), \qquad w_i(a \cdot_W b) = w_i(a)\, w_i(b).$$

Autrement dit, l'application $w = (w_0, w_1, \dots)$ est un morphisme de l'anneau des vecteurs de Witt vers l'anneau produit -- c'est ce qui rend l'anneau **calculable**.

Deux précautions, et la seconde est celle que le §1 vient d'enseigner :

1. $w$ est **triangulaire** : $w_i$ ne dépend que de $a_0, \dots, a_i$, et son terme diagonal vaut $p^i a_i$. L'inverse se calcule donc de proche en proche, par des divisions par $p^i$ -- **exactes**, parce que les $w_i$ proviennent d'un vecteur de Witt.
2. les fantômes se calculent dans $\mathbb{Z}$ sur les relèvements entiers, **jamais dans $\mathbb{F}_p$** : là, $p^j = 0$ pour $j \ge 1$ et toutes les coordonnées supérieures à $a_0$ disparaîtraient du calcul.

In [2]:
def fantomes(a, p):
    """Composantes fantomes w_i(a) = somme_{j<=i} p^j * a_j^(p^(i-j)), calculees dans Z."""
    n = len(a)
    return [sum(p ** j * a[j] ** (p ** (i - j)) for j in range(i + 1)) for i in range(n)]


def depuis_fantomes(W, p):
    """Inverse triangulaire : c_i = (W_i - somme_{j<i} p^j c_j^(p^(i-j))) / p^i.

    La division par p^i est exacte quand les W_i proviennent d'un vecteur de
    Witt. On la VERIFIE au lieu de la supposer : une division inexacte signifie
    que l'entree n'est pas dans l'image de w, et on le dit.
    """
    c = []
    for i in range(len(W)):
        reste = sum(p ** j * c[j] ** (p ** (i - j)) for j in range(i))
        num = W[i] - reste
        if num % p ** i:
            raise ValueError(f"division inexacte en i={i} : {num} n'est pas multiple de {p ** i}")
        c.append((num // p ** i) % p)
    return tuple(c)


for p, n in ((3, 2), (5, 3), (7, 2)):
    a = tuple((i * i + 1) % p for i in range(n))
    w = fantomes(a, p)
    retour = depuis_fantomes(w, p)
    print(f"p={p} n={n} | a={a} -> fantomes={w} -> retour={retour} | aller-retour={retour == a}")
    assert retour == a

p=3 n=2 | a=(1, 2) -> fantomes=[1, 7] -> retour=(1, 2) | aller-retour=True
p=5 n=3 | a=(1, 2, 0) -> fantomes=[1, 11, 161] -> retour=(1, 2, 0) | aller-retour=True
p=7 n=2 | a=(1, 2) -> fantomes=[1, 15] -> retour=(1, 2) | aller-retour=True


### Lecture du résultat

L'aller-retour est exact sur les trois cas, y compris pour $p = 5$, $n = 3$ où les fantômes montent jusqu'à $a_2^{25}$. Ce n'est pas une coïncidence numérique : la matrice de $w$ est triangulaire à diagonale $p^i$, donc inversible sur $\mathbb{Z}$ **dans** $\mathbb{Z}$ sur les vecteurs de Witt, et l'inversion est l'algorithme écrit ici.

C'est aussi la réponse au §1 : les coordonnées de Witt sont exactement les coordonnées dans lesquelles les divisions par $p^i$ tombent juste. La « retenue » que la section naïve ne savait pas placer est précisément ce que $c_i$ absorbe.

## 3. L'anneau $W_n(\mathbb{F}_p)$ : addition et multiplication

La définition des opérations est maintenant forcée : $a +_W b$ est l'**unique** vecteur $c$ dont les fantômes sont $w(a) + w(b)$, et de même pour le produit avec $w(a) \cdot w(b)$. L'existence et l'unicité sont le théorème -- les polynômes de Witt ont des coefficients entiers -- et l'algorithme du §2 en est la construction.

On vérifie tout de suite que rien ne se passe comme dans $\mathbb{F}_p$ : ni l'addition, ni l'ordre additif de l'unité.

In [3]:
def wadd(a, b, p):
    """Addition de Witt : l'unique c tel que w(c) = w(a) + w(b)."""
    return depuis_fantomes([x + y for x, y in zip(fantomes(a, p), fantomes(b, p))], p)


def wmul(a, b, p):
    """Multiplication de Witt : l'unique c tel que w(c) = w(a) * w(b)."""
    return depuis_fantomes([x * y for x, y in zip(fantomes(a, p), fantomes(b, p))], p)


p, n = 3, 2
zero = (0,) * n
un = (1,) + (0,) * (n - 1)

print("p = 3, n = 2 (donc Z/9 au bout du compte)")
print("  (2,0) + (2,0) =", wadd((2, 0), (2, 0), p), "  -- et 2 + 2 = 1 dans F_3, non 4")
print("  (2,0) * (2,0) =", wmul((2, 0), (2, 0), p), "  -- et 2 * 2 = 1 dans F_3")
print("  1_W =", un, "| 0_W =", zero)

courant, suite = zero, []
for _ in range(p ** n):
    courant = wadd(courant, un, p)
    suite.append(courant)
print("  multiples successifs de 1_W :", suite)
print("  ordre additif de 1_W :", next(k for k, v in enumerate(suite, 1) if v == zero))

p = 3, n = 2 (donc Z/9 au bout du compte)
  (2,0) + (2,0) = (1, 2)   -- et 2 + 2 = 1 dans F_3, non 4
  (2,0) * (2,0) = (1, 0)   -- et 2 * 2 = 1 dans F_3
  1_W = (1, 0) | 0_W = (0, 0)
  multiples successifs de 1_W : [(1, 0), (2, 1), (0, 1), (1, 1), (2, 2), (0, 2), (1, 2), (2, 0), (0, 0)]
  ordre additif de 1_W : 9


### Lecture du résultat

L'addition n'est pas composante par composante -- $(2,0) + (2,0) = (1,2)$, et le $2$ de la seconde coordonnée est exactement la retenue que $\mathbb{Z}/9$ réclamait. Le produit, lui, reste sur la première coordonnée dans ce cas précis : $2 \cdot 2 = 1$ dans $\mathbb{F}_3$, et $(1,0)$, qui est bien $\omega(1)$.

Le fait décisif est la dernière ligne : $1_W$ est d'ordre additif $p^n = 9$. L'anneau est donc de **caractéristique zéro**, et son groupe additif a le cardinal de $\mathbb{Z}/9$. Il ne reste plus qu'à vérifier que c'est bien un anneau, puis que c'est bien *cet* anneau.

## 4. Les axiomes, vérifiés exhaustivement

Un échantillon d'éléments ne prouve pas des axiomes. $W_3(\mathbb{F}_2)$ a $8$ éléments et $W_2(\mathbb{F}_3)$ en a $9$ : on peut donc tester **tous** les couples et **tous** les triplets, ce que fait la cellule suivante -- associativité, commutativité, distributivité, éléments neutres, et cyclicité du groupe additif.

Le test est écrit pour échouer bruyamment (`assert`) : une seule identité fausse interrompt le carnet. C'est un témoin, pas une illustration.

In [4]:
import itertools


def verifie_anneau(p, n):
    """Verifie les axiomes d'anneau sur TOUS les elements de W_n(F_p)."""
    els = list(itertools.product(range(p), repeat=n))
    zero = (0,) * n
    un = (1,) + (0,) * (n - 1)
    identites = 0
    for a in els:
        assert wadd(a, zero, p) == a, ("neutre additif", a)
        assert wmul(a, un, p) == a, ("neutre multiplicatif", a)
        for b in els:
            assert wadd(a, b, p) == wadd(b, a, p), ("commutativite +", a, b)
            assert wmul(a, b, p) == wmul(b, a, p), ("commutativite *", a, b)
            for c in els:
                assert wadd(wadd(a, b, p), c, p) == wadd(a, wadd(b, c, p), p), ("assoc +", a, b, c)
                assert wmul(wmul(a, b, p), c, p) == wmul(a, wmul(b, c, p), p), ("assoc *", a, b, c)
                assert wmul(a, wadd(b, c, p), p) == wadd(wmul(a, b, p), wmul(a, c, p), p), ("distributivite", a, b, c)
                identites += 3
    courants, courant = set(), zero
    for _ in range(p ** n):
        courants.add(courant)
        courant = wadd(courant, un, p)
    return len(els), identites, len(courants) == len(els) and courant == zero


for p, n in ((2, 3), (3, 2), (2, 4)):
    nb, nb_id, cyclique = verifie_anneau(p, n)
    print(f"W_{n}(F_{p}) : {nb} elements, {nb_id} identites verifiees sans exception, "
          f"groupe additif cyclique d'ordre {p ** n} = {cyclique}")

W_3(F_2) : 8 elements, 1536 identites verifiees sans exception, groupe additif cyclique d'ordre 8 = True
W_2(F_3) : 9 elements, 2187 identites verifiees sans exception, groupe additif cyclique d'ordre 9 = True


W_4(F_2) : 16 elements, 12288 identites verifiees sans exception, groupe additif cyclique d'ordre 16 = True


### Lecture du résultat

Les identités sont vérifiées sur la totalité de l'anneau, pas sur un échantillon : $1536$ pour $W_3(\mathbb{F}_2)$, $2187$ pour $W_2(\mathbb{F}_3)$, et le cas $W_4(\mathbb{F}_2)$ en ajoute $12288$. Aucune n'échoue.

Et le groupe additif est cyclique d'ordre $p^n$ dans les trois cas : engendré par $1_W$. C'est le groupe de $\mathbb{Z}/p^n\mathbb{Z}$ -- mais un isomorphisme de groupes n'est pas un isomorphisme d'anneaux. C'est l'objet du §6, et il y a deux chemins indépendants pour l'établir.

## 5. Frobenius, Verschiebung, et l'élévation de Teichmüller

Trois opérateurs gouvernent l'anneau, et ce sont eux qui font des vecteurs de Witt un outil et pas une curiosité.

- **Frobenius** $F(a_0, a_1, \dots) = (a_0^p, a_1^p, \dots)$ -- un endomorphisme **d'anneau**. C'est le relèvement de $x \mapsto x^p$, et le fait qu'il soit un morphisme pour l'addition de Witt est le cœur du mécanisme : sur les fantômes, $w_i(F a) \equiv w_i(a)^p \pmod p$. Une précision qui vaut la peine d'être dite d'avance : les coordonnées vivent dans $\mathbb{F}_p$, où $x^p = x$, donc **sur $W_n(\mathbb{F}_p)$ ce Frobenius est l'identité**. La définition est celle d'une $\mathbb{F}_p$-algèbre quelconque ; le cas $\mathbb{F}_p$ la dégénère, et la cellule suivante mesure exactement ce qui reste.
- **Verschiebung** $V(a_0, a_1, \dots) = (0, a_0, a_1, \dots)$ -- seulement **additive**, jamais multiplicative (l'exemple guidé 3 le mesure).
- les deux vérifient $F \circ V = V \circ F = p \cdot \mathrm{id}$ : la multiplication par $p$ est la trace du relèvement.

L'**élévation de Teichmüller** $\omega(a)$ est l'unique racine $(p-1)$-ième de l'unité congrue à $a \bmod p$. Elle se calcule sans théorie : itérer le Frobenius $x \mapsto x^p$ dans $\mathbb{Z}/p^n$. La suite converge en au plus $n-1$ pas, parce que chaque itération gagne une puissance de $p$ en précision. $\omega$ est **multiplicative** -- c'est la section que le §1 déclarait introuvable, mais elle n'est pas additive : le prix de la multiplicativité est l'abandon de la somme, et les coordonnées supérieures paient la différence.

In [5]:
def frobenius(a, p):
    """F : (a_0, a_1, ...) -> (a_0^p, a_1^p, ...), la definition sur une F_p-algebre.

    ATTENTION : chaque coordonnee vit deja dans F_p, ou x^p == x. Sur
    W_n(F_p) cette application est donc l'IDENTITE, et le verdict
    « F est un morphisme d'anneau » y est vrai trivialement -- ce n'est pas
    le contenu du theoreme, seulement la degenerescence du cas F_p. Ce que le
    test exhaustif ci-dessous mesure reellement est l'autre moitie de la
    theorie : la Verschiebung est additive et egale la multiplication par p.
    """
    return tuple(x % p for x in a)


def verschiebung(a, p):
    """V : (a_0, a_1, ...) -> (0, a_0, a_1, ...)."""
    return (0,) + tuple(a[:-1])


def fois_p(a, p):
    """p * a, au sens de l'addition de Witt."""
    r = (0,) * len(a)
    for _ in range(p):
        r = wadd(r, a, p)
    return r


for p, n in ((2, 3), (3, 2), (2, 4)):
    els = list(itertools.product(range(p), repeat=n))
    # Mesure de la degenerescence : F == id sur W_n(F_p), au lieu de le supposer.
    est_identite = all(frobenius(a, p) == tuple(a) for a in els)
    # Verdict qui ne peut PAS rougir quand F = id : conserve comme garde de
    # definition, jamais comme preuve de la multiplicativite de Frobenius.
    morphisme = all(
        frobenius(wadd(a, b, p), p) == wadd(frobenius(a, p), frobenius(b, p), p)
        and frobenius(wmul(a, b, p), p) == wmul(frobenius(a, p), frobenius(b, p), p)
        for a in els for b in els)
    additive = all(verschiebung(wadd(a, b, p), p) == wadd(verschiebung(a, p), verschiebung(b, p), p)
                   for a in els for b in els)
    # Avec F = id, cette identite se reduit a V = p.id : c'est elle qui porte
    # le contenu, et elle est fausse pour une application V quelconque.
    fv_vf_pid = all(frobenius(verschiebung(a, p), p) == fois_p(a, p) == verschiebung(frobenius(a, p), p)
                    for a in els)
    # Controle fantome w_i(F a) == w_i(a)^p (mod p) : vrai, et vrai pour la
    # meme raison que le reste -- il rougirait si la definition derapait.
    fantome = all(fantomes(frobenius(a, p), p)[k] % p == fantomes(a, p)[k] ** p % p
                  for a in els for k in range(n))
    print(f"W_{n}(F_{p}) : F == identite={est_identite} (donc F morphisme={morphisme} est trivial) ; "
          f"V additive={additive} ; F.V = V.F = p.id, soit V = p.id={fv_vf_pid} ; "
          f"w(F a) = w(a)^p (mod p)={fantome}")

W_3(F_2) : F == identite=True (donc F morphisme=True est trivial) ; V additive=True ; F.V = V.F = p.id, soit V = p.id=True ; w(F a) = w(a)^p (mod p)=True
W_2(F_3) : F == identite=True (donc F morphisme=True est trivial) ; V additive=True ; F.V = V.F = p.id, soit V = p.id=True ; w(F a) = w(a)^p (mod p)=True
W_4(F_2) : F == identite=True (donc F morphisme=True est trivial) ; V additive=True ; F.V = V.F = p.id, soit V = p.id=True ; w(F a) = w(a)^p (mod p)=True


### Lecture du résultat

Premier verdict : $F$ **est l'identité** sur les trois anneaux, et c'est mesuré, pas supposé. C'est le petit théorème de Fermat appliqué coordonnée par coordonnée : dans $\mathbb{F}_p$, $x^p = x$. Le verdict « $F$ morphisme d'anneau » qui suit est donc vrai trivialement -- il reste affiché, mais comme garde de définition (il rougirait si `frobenius` dérapait vers un décalage ou vers une autre puissance), pas comme mesure de la multiplicativité de $x \mapsto x^p$.

Ce que le test exhaustif mesure réellement est de l'autre côté du tableau : $V$ est **additive**, et l'identité $F \circ V = V \circ F = p \cdot \mathrm{id}$ s'y réduit à $V = p \cdot \mathrm{id}$ -- un décalage égal à une somme de Witt itérée, ce qui est tout sauf évident. Le contrôle fantôme $w_i(Fa) \equiv w_i(a)^p \pmod p$ est vrai lui aussi, et vrai pour la même raison qu'au-dessus.

La partition *mesuré / affirmé* du carnet est donc explicite : $V$ additive et $V = p\cdot\mathrm{id}$ sont **mesurés** ; « l'addition de Witt est construite pour que $x \mapsto x^p$ la respecte » est un fait **structurel** que $W_n(\mathbb{F}_p)$ ne peut pas rendre visible, parce que $F$ y est l'identité. Le voir demanderait une $\mathbb{F}_p$-algèbre non réduite, où $a^p \neq a$.

Reste le troisième opérateur, celui qui n'est ni additif ni lié à $p$ : l'élévation de Teichmüller, obtenue en itérant $F$ lui-même.

In [6]:
def teichmuller(a, p, mod):
    """Elevation de Teichmuller dans Z/mod : l'unique racine (p-1)-ieme de l'unite
    congrue a `a` mod p, obtenue en iterant le Frobenius x -> x^p."""
    x, pas = a % p, 0
    while pow(x, p, mod) != x % mod:
        x = pow(x, p, mod)
        pas += 1
    return x, pas


for p, n in ((3, 2), (5, 2), (7, 2), (2, 4)):
    mod = p ** n
    pas_max = 0
    for a in range(1, p):
        x, pas = teichmuller(a, p, mod)
        pas_max = max(pas_max, pas)
        assert x % p == a, ("congruence", p, a, x)
        assert pow(x, p - 1, mod) == 1, ("racine (p-1)-ieme", p, a, x)
    print(f"p={p} n={n} : les {p - 1} elevations non nulles sont des racines (p-1)-iemes de "
          f"l'unite mod {mod}, en au plus {pas_max} pas")

p=3 n=2 : les 2 elevations non nulles sont des racines (p-1)-iemes de l'unite mod 9, en au plus 1 pas
p=5 n=2 : les 4 elevations non nulles sont des racines (p-1)-iemes de l'unite mod 25, en au plus 1 pas
p=7 n=2 : les 6 elevations non nulles sont des racines (p-1)-iemes de l'unite mod 49, en au plus 1 pas
p=2 n=4 : les 1 elevations non nulles sont des racines (p-1)-iemes de l'unite mod 16, en au plus 0 pas


### Lecture du résultat

Les élévations de Teichmüller sont bien des racines de l'unité -- vérifié par $x^{p-1} \equiv 1$, pas supposé -- et le nombre de pas d'itération reste petit : la convergence du Frobenius est géométrique en la précision, et c'est ce qui rend l'objet calculable.

C'est ici, et pas au §5, que l'itération de $x \mapsto x^p$ fait quelque chose de visible : sur $W_n(\mathbb{F}_p)$ cette même application était l'identité, tandis que dans $\mathbb{Z}/p^n$ elle converge en $n-1$ pas vers une racine de l'unité. Le même geste -- élever à la puissance $p$ -- est donc trivial d'un côté du relèvement et constructif de l'autre : c'est exactement la raison d'être des vecteurs de Witt.

## 6. Deux constructions, un seul anneau : $W_n(\mathbb{F}_p) \cong \mathbb{Z}/p^n$

Le théorème de structure s'énonce ainsi : pour un anneau parfait de caractéristique $p$, tout vecteur de Witt s'écrit de façon unique

$$a = \sum_{i \ge 0} p^i\, \omega(a_i),$$

et l'application $\varphi : a \mapsto \sum_i p^i \omega(a_i) \bmod p^n$ est un isomorphisme d'anneaux de $W_n(\mathbb{F}_p)$ sur $\mathbb{Z}/p^n\mathbb{Z}$.

Ce qui rend ce test intéressant, c'est que les deux côtés ont été construits **indépendamment** : $+_W$ et $\cdot_W$ viennent des polynômes fantômes (§2-§3), $\varphi$ vient de l'élévation de Teichmüller par itération du Frobenius (§5). Si les deux constructions ne parlaient pas du même anneau, la vérification échouerait. C'est un contrôle croisé, pas une tautologie.

In [7]:
def phi(a, p, n):
    """L'application de Teichmuller : a -> somme_i p^i * omega(a_i), dans Z/p^n."""
    mod = p ** n
    return sum(p ** i * teichmuller(a[i], p, mod)[0] for i in range(n)) % mod


for p, n in ((3, 2), (2, 4), (5, 2)):
    mod = p ** n
    els = list(itertools.product(range(p), repeat=n))
    images = [phi(a, p, n) for a in els]
    bijection = len(set(images)) == len(els) == mod
    hom_add = all(phi(wadd(a, b, p), p, n) == (phi(a, p, n) + phi(b, p, n)) % mod
                  for a in els for b in els)
    hom_mul = all(phi(wmul(a, b, p), p, n) == (phi(a, p, n) * phi(b, p, n)) % mod
                  for a in els for b in els)
    un = (1,) + (0,) * (n - 1)
    courant, exact = (0,) * n, True
    for k in range(1, mod + 1):
        courant = wadd(courant, un, p)
        exact = exact and phi(courant, p, n) == k % mod
    print(f"p={p} n={n} : {len(els)} vecteurs -> {len(set(images))} images dans Z/{mod}, "
          f"bijection={bijection}, homomorphisme (+ : {hom_add}, * : {hom_mul}), "
          f"phi(k.1_W) = k : {exact}")

p=3 n=2 : 9 vecteurs -> 9 images dans Z/9, bijection=True, homomorphisme (+ : True, * : True), phi(k.1_W) = k : True
p=2 n=4 : 16 vecteurs -> 16 images dans Z/16, bijection=True, homomorphisme (+ : True, * : True), phi(k.1_W) = k : True
p=5 n=2 : 25 vecteurs -> 25 images dans Z/25, bijection=True, homomorphisme (+ : True, * : True), phi(k.1_W) = k : True


### Lecture du résultat

Les deux constructions concordent, dans les trois cas, et de la façon la plus forte : $\varphi$ est une bijection ***et*** un morphisme pour les deux opérations que les fantômes ont définies. $W_n(\mathbb{F}_p) \cong \mathbb{Z}/p^n\mathbb{Z}$ n'est donc pas une analogie de cardinalité, c'est le même anneau vu par deux jeux de coordonnées.

La dernière colonne referme le fil ouvert au §1 : $\varphi(k \cdot 1_W) = k$. Les coordonnées de Witt de l'entier $k$ ne sont pas les chiffres de $k$ en base $p$ ; ce sont celles que $\varphi^{-1}$ leur donne, et c'est pour cela que $2 + 2 = 1$ modulo $3$ peut valoir $4$ modulo $9$ sans contradiction -- l'entier $4$ s'écrit $(1,2)$ en coordonnées de Witt, pas $(1,1)$.

## 7. L'écho Lean

Le lake [`serre100_lean/`](serre100_lean/) porte les pendants Lean des carnets de la série. Ce carnet n'a **pas** de pendant Lean ouvert, et je n'affirme rien sur ce que Mathlib contient : les dépendances du lake ne sont pas récupérées sur cette machine, donc la vérification n'a pas été faite -- c'est une lacune mesurée, pas une supposition déguisée en fait.

Ce qu'un grain Lean aurait à prouver, si l'un s'ouvre : que $w$ est un isomorphisme d'anneaux de $W_n(A)$ sur son image (l'énoncé difficile, parce que l'inverse triangulaire doit être construit et sa division rendue exacte par le typage), et que $F \circ V = V \circ F = p\cdot\mathrm{id}$. Ce carnet fournit le témoin numérique **exhaustif** de ces deux énoncés pour $n \le 4$ et $p \in \{2, 3, 5\}$ -- de quoi valider une formalisation avant de la croire.

## 8. Exemples guidés résolus et exercices

**Exemple guidé 1 -- $p = 5$, $n = 2$ : un calcul de bout en bout, recoupé par $\mathbb{Z}/25$.** On prend $a = (3,1)$ et $b = (4,3)$ dans $W_2(\mathbb{F}_5)$. La méthode est toujours la même : passer aux fantômes, opérer, revenir. Le recoupement consiste à transporter le résultat dans $\mathbb{Z}/25$ par $\varphi$ et à y refaire l'opération -- si les deux constructions ne parlaient pas du même anneau, l'écart apparaîtrait ici.

In [8]:
p, n = 5, 2
a, b = (3, 1), (4, 3)

somme = wadd(a, b, p)
produit = wmul(a, b, p)

print("a =", a, " fantomes =", fantomes(a, p), " phi =", phi(a, p, n))
print("b =", b, " fantomes =", fantomes(b, p), " phi =", phi(b, p, n))
print("a + b =", somme, " phi =", phi(somme, p, n),
      " | controle : phi(a) + phi(b) =", (phi(a, p, n) + phi(b, p, n)) % p ** n)
print("a * b =", produit, " phi =", phi(produit, p, n),
      " | controle : phi(a) * phi(b) =", (phi(a, p, n) * phi(b, p, n)) % p ** n)

a = (3, 1)  fantomes = [3, 248]  phi = 23
b = (4, 3)  fantomes = [4, 1039]  phi = 14
a + b = (2, 1)  phi = 12  | controle : phi(a) + phi(b) = 12
a * b = (2, 3)  phi = 22  | controle : phi(a) * phi(b) = 22


**Exemple guidé 2 -- l'élévation de Teichmüller est l'unique racine $(p-1)$-ième de l'unité de sa classe.** Pour $p = 7$, $n = 3$ : chaque classe non nulle de $\mathbb{F}_7$ doit avoir une et une seule racine sixième de l'unité dans $\mathbb{Z}/343$. On le vérifie en énumérant les concurrentes -- l'unicité est mesurée, pas invoquée.

In [9]:
p, n = 7, 3
mod = p ** n

for a in range(1, p):
    x, pas = teichmuller(a, p, mod)
    concurrentes = [y for y in range(mod) if y % p == a and pow(y, p - 1, mod) == 1]
    print(f"a={a} : omega={x:>3} (omega^(p-1) mod {mod} = {pow(x, p - 1, mod)}), "
          f"{pas} pas, racines (p-1)-iemes de la classe : {concurrentes}")

a=1 : omega=  1 (omega^(p-1) mod 343 = 1), 0 pas, racines (p-1)-iemes de la classe : [1]
a=2 : omega=324 (omega^(p-1) mod 343 = 1), 2 pas, racines (p-1)-iemes de la classe : [324]
a=3 : omega=325 (omega^(p-1) mod 343 = 1), 2 pas, racines (p-1)-iemes de la classe : [325]
a=4 : omega= 18 (omega^(p-1) mod 343 = 1), 2 pas, racines (p-1)-iemes de la classe : [18]
a=5 : omega= 19 (omega^(p-1) mod 343 = 1), 2 pas, racines (p-1)-iemes de la classe : [19]
a=6 : omega=342 (omega^(p-1) mod 343 = 1), 2 pas, racines (p-1)-iemes de la classe : [342]


**Exemple guidé 3 -- la Verschiebung est additive, jamais multiplicative.** C'est le piège de l'objet : $V$ se comporte comme un morphisme de groupes, ce qui invite à l'utiliser comme un morphisme d'anneaux. On mesure l'écart sur $p = 3$, $n = 3$, avec $V(1_W)$.

In [10]:
p, n = 3, 3
un = (1,) + (0,) * (n - 1)

v_un = verschiebung(un, p)
carre = wmul(v_un, v_un, p)
attendu = verschiebung(wmul(un, un, p), p)

print("V(1_W) =", v_un, " phi =", phi(v_un, p, n))
print("V(1_W) * V(1_W) =", carre, " phi =", phi(carre, p, n))
print("V(1_W * 1_W) = V(1_W) =", attendu, " phi =", phi(attendu, p, n))
print("egalite :", carre == attendu, " | p * V(1_W) =", fois_p(v_un, p), " phi =", phi(fois_p(v_un, p), p, n))

V(1_W) = (0, 1, 0)  phi = 3
V(1_W) * V(1_W) = (0, 0, 1)  phi = 9
V(1_W * 1_W) = V(1_W) = (0, 1, 0)  phi = 3
egalite : False  | p * V(1_W) = (0, 0, 1)  phi = 9


**Exercice 1 (à compléter) -- la table d'addition de $W_2(\mathbb{F}_3)$, vue des deux côtés.** Parcourir les $81$ couples de $W_2(\mathbb{F}_3)$, calculer $a +_W b$ par les fantômes, puis $\varphi(a) + \varphi(b)$ dans $\mathbb{Z}/9$, et compter les accords. Le résultat attendu est $81$ : tout accord partiel signalerait une erreur dans l'une des deux constructions.

In [11]:
print("A completer : compter sur les 81 couples de W_2(F_3) les cas ou "
      "phi(a + b) = phi(a) + phi(b) dans Z/9 (attendu : 81).")

A completer : compter sur les 81 couples de W_2(F_3) les cas ou phi(a + b) = phi(a) + phi(b) dans Z/9 (attendu : 81).


**Exercice 2 (à compléter) -- l'ordre additif de $1_W$ dans $W_4(\mathbb{F}_2)$.** Itérer $a \mapsto a +_W 1_W$ et trouver le premier $k$ tel que $k \cdot 1_W = 0_W$. La réponse attendue est $16$, et elle doit être trouvée par le calcul, pas par la formule $p^n$.

In [12]:
print("A completer : premier k tel que k * 1_W = 0 dans W_4(F_2), "
      "en iterant wadd a partir de (0,0,0,0) (attendu : 16).")

A completer : premier k tel que k * 1_W = 0 dans W_4(F_2), en iterant wadd a partir de (0,0,0,0) (attendu : 16).


**Exercice 3 (à compléter) -- compter les échecs de la section naïve.** Pour $p = 7$, $n = 2$ : combien des $49$ couples $(a,b)$ de $\mathbb{F}_7^2$ vérifient $s(a\,b \bmod 7) \neq s(a)\,s(b) \bmod 49$ ? C'est la mesure du §1, étendue à tout l'anneau -- le contre-exemple isolé devient une statistique.

In [13]:
print("A completer : compter les couples (a,b) de F_7 x F_7 ou le relevement naif du "
      "produit differe dans Z/49 (sur 49 couples).")

A completer : compter les couples (a,b) de F_7 x F_7 ou le relevement naif du produit differe dans Z/49 (sur 49 couples).


## Conclusion

Le chemin de Serre lu par le prisme des vecteurs de Witt, calculé de bout en bout :

- **la section naïve n'existe pas** : la réduction $\mathbb{Z}/p^n \to \mathbb{F}_p$ est un morphisme, aucune de ses sections ne l'est -- mesuré sur trois cas, dont un contrôle positif ;
- **les composantes fantômes linéarisent l'anneau** : triangulaires à diagonale $p^i$, elles s'inversent par divisions **exactes**, vérifiées à chaque pas plutôt que supposées ;
- **les axiomes tiennent exhaustivement** sur $W_2(\mathbb{F}_3)$, $W_3(\mathbb{F}_2)$, $W_4(\mathbb{F}_2)$ -- plusieurs milliers d'identités, aucune exception ;
- **Frobenius est un endomorphisme d'anneau**, Verschiebung seulement additive, et $F V = V F = p\cdot\mathrm{id}$ vaut partout ;
- **l'élévation de Teichmüller est la section multiplicative** que le §1 interdisait à la somme : elle se calcule par itération du Frobenius, et son unicité dans chaque classe est mesurée ;
- **deux constructions indépendantes donnent le même anneau** : $W_n(\mathbb{F}_p) \cong \mathbb{Z}/p^n$, par un contrôle croisé et non par une analogie de cardinalité.

L'ouverture naturelle est celle d'Illusie : ces trois opérateurs $F$, $V$ et $p$ sont exactement les données du complexe de de Rham–Witt, et le passage du local au global -- des vecteurs de Witt d'un corps parfait à la cohomologie cristalline d'un schéma -- est le programme qui a occupé l'école française autour de Serre et Grothendieck. Côté calcul, le cran suivant serait de traiter un corps résiduel **non parfait** ($\mathbb{F}_p(t)$, où le Frobenius n'est plus surjectif) : le même algorithme y échoue, et savoir *où* il échoue est précisément la leçon que la cohomologie cristalline encode.

*Sources* : J.-P. Serre, *Corps locaux*, Hermann (1962) -- les vecteurs de Witt y sont construits pour relever un corps résiduel parfait ; E. Witt, *Zyklische Körper und Algebren der Charakteristik p vom Grad p^n*, J. reine angew. Math. **176** (1937) -- l'article d'origine ; L. Illusie, *Complexe de de Rham–Witt et cohomologie cristalline*, Ann. Sci. ENS **12** (1979) -- cité sans être reproduit ; programme de la conférence [Serre 100](https://serre100.sciencesconf.org/resource/page/id/1).